In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

In [2]:
train_data = pd.read_csv("../tmp/titanic/train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test_data = pd.read_csv("../tmp/titanic/test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [4]:
train_data.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [5]:
# Gán giá trị mode (giá trị xuất hiện nhiều nhất) cho các giá trị missing của cột "Embarked" 
most_common_embarked = train_data['Embarked'].mode()[0]
train_data['Embarked'] = train_data['Embarked'].fillna(most_common_embarked)

# Gán giá trị trung bình cho các giá trị  missing của cột "Age"
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].median())

# Xóa cột "Cabin" khỏi dataFrame vì quá nhiều hàng missing
train_data.drop(columns=['Cabin'], inplace=True)
test_data.drop(columns=['Cabin'], inplace=True)

train_data.drop(columns=['PassengerId'], inplace=True)
test_data.drop(columns=['PassengerId'], inplace=True)

In [6]:
test_data.isna().sum()

Pclass       0
Name         0
Sex          0
Age         86
SibSp        0
Parch        0
Ticket       0
Fare         1
Embarked     0
dtype: int64

In [7]:
test_data['Age'] = test_data['Age'].fillna(test_data['Age'].median())
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].median())

In [8]:
print(train_data.isna().sum().sum())
print(test_data.isna().sum().sum())

0
0


In [9]:
# Map male - 0, fale - 1
train_data['Sex'] = train_data['Sex'].map({'male': 0, 'female': 1})
test_data['Sex'] = test_data['Sex'].map({'male': 0, 'female': 1})

train_data = pd.get_dummies(train_data, columns=['Embarked'], drop_first=True)
test_data = pd.get_dummies(test_data, columns=['Embarked'], drop_first=True)

train_data = pd.get_dummies(train_data, columns=['Pclass'], drop_first=True)
test_data = pd.get_dummies(test_data, columns=['Pclass'], drop_first=True)

In [10]:
# 1. Tạo name_title từ Name (làm trước khi drop)
train_data['name_title'] = train_data.Name.apply(lambda x: x.split(',')[1].split('.')[0].strip())
test_data['name_title'] = test_data.Name.apply(lambda x: x.split(',')[1].split('.')[0].strip())

# 2. Lấy top 5 title phổ biến nhất
top_5_titles = train_data['name_title'].value_counts().head(5).index.tolist()
print("Top 5 titles:", top_5_titles)

# 3. Gom tất cả title khác thành "Other"
train_data['name_title'] = train_data['name_title'].apply(lambda x: x if x in top_5_titles else 'Other')
test_data['name_title'] = test_data['name_title'].apply(lambda x: x if x in top_5_titles else 'Other')

# 4. Kiểm tra kết quả - CHỈ CÒN 6 CATEGORIES
print("\nAfter grouping - chỉ còn 6 categories:")
print(train_data['name_title'].value_counts())
print("\nTest data name_title counts:")
print(test_data['name_title'].value_counts())

# 5. Sau đó mới drop column Name
train_data.drop(columns=['Name'], inplace=True)
test_data.drop(columns=['Name'], inplace=True)

train_data = pd.get_dummies(train_data, columns=['name_title'], drop_first=True)
test_data = pd.get_dummies(test_data, columns=['name_title'], drop_first=True)

train_data.drop(columns=['Ticket'], inplace=True)
test_data.drop(columns=['Ticket'], inplace=True)


Top 5 titles: ['Mr', 'Miss', 'Mrs', 'Master', 'Dr']

After grouping - chỉ còn 6 categories:
name_title
Mr        517
Miss      182
Mrs       125
Master     40
Other      20
Dr          7
Name: count, dtype: int64

Test data name_title counts:
name_title
Mr        240
Miss       78
Mrs        72
Master     21
Other       6
Dr          1
Name: count, dtype: int64


In [11]:
# train_data['name_title'].value_counts()

In [12]:
train_data['Fare'] = np.log1p(train_data['Fare'].clip(lower=0))
test_data['Fare'] = np.log1p(test_data['Fare'].clip(lower=0))

In [13]:
train_data.columns

Index(['Survived', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Q',
       'Embarked_S', 'Pclass_2', 'Pclass_3', 'name_title_Master',
       'name_title_Miss', 'name_title_Mr', 'name_title_Mrs',
       'name_title_Other'],
      dtype='object')

In [14]:
sum([1 for _ in train_data.columns])

15

In [15]:
fdsfdsfdsfdsf

NameError: name 'fdsfdsfdsfdsf' is not defined

In [ ]:
fd

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
for col in ['Age', 'Fare']:
    train_data[col] = scaler.fit_transform(train_data[[col]])
    test_data[col] = scaler.transform(test_data[[col]])

In [ ]:
from sklearn.model_selection import train_test_split

X = train_data.drop('Survived', axis=1)
y = train_data['Survived']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1, n_estimators=200),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:,1]

    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    roc_auc = roc_auc_score(y_val, y_prob)

    print(f"{name} Results:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  ROC AUC:  {roc_auc:.4f}")
    print("-" * 30)

Logistic Regression Results:
  Accuracy: 0.8436
  F1 Score: 0.7879
  ROC AUC:  0.8701
------------------------------
Random Forest Results:
  Accuracy: 0.8268
  F1 Score: 0.7669
  ROC AUC:  0.8375
------------------------------
XGBoost Results:
  Accuracy: 0.8156
  F1 Score: 0.7591
  ROC AUC:  0.8385
------------------------------


In [ ]:
best_model = LogisticRegression(random_state=42, max_iter=1000)
best_model.fit(X_train, y_train)

y_val_pred = best_model.predict(X_val)

print("Actual values (first 5):")
print(y_val.head())

print("\nPredicted values (first 5):")
print(pd.Series(y_val_pred).head())

Actual values (first 5):
565    0
160    0
553    1
860    0
241    1
Name: Survived, dtype: int64

Predicted values (first 5):
0    0
1    0
2    0
3    0
4    1
dtype: int64


In [ ]:
# Orijinal test csv'sini tekrar yükle (PassengerId için)
test_data_orig = pd.read_csv("/kaggle/input/titanic/test.csv")

# best_model ile test verisi üzerinde tahmin yap
test_preds = best_model.predict(test_data)

# Submission dataframe'i oluştur
submission = pd.DataFrame({
    "PassengerId": test_data_orig["PassengerId"],
    "Survived": test_preds
})

# CSV olarak kaydet (kaggle ortamında)
submission.to_csv("submission.csv", index=False)

print("Submission file 'submission.csv' created.")

Submission file 'submission.csv' created.
